# Signed Jumps 01 — volatility engine + signed Hawkes jump layer (Binance spot 1m)

Protocol: `discussions/Signed_Jumps/001_claude_sj01_preregistration.md` (fixed before this run).

Ladder: (a) Poisson → (b) engine-modulated Poisson (null) → (c) excitation → (d) parent sign matters → (e) + jump-truncated skewness.
Gates on TEST (last 30% of days): **G1** (c)>(b), **G2a** (d)>(c), **G2b** (e)>(d) [Bonferroni], **G3** 15-minute signed jump forecast vs (b).

In [ ]:
import os,sys,json,time,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import signed_jumps as sj
importlib.reload(sj)
assert sj.VERSION=='SJ01-0.1.0'
SYNTHETIC=os.getenv('SJ01_SYNTHETIC','0')=='1'
p=sj.Params()
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='sj01-');OUT=Path(tmp.name)/'01'
    p=sj.Params(n_boot=200,half_lives=(3,10),min_test_jumps_per_sign=20,mc_paths=100,c_sensitivity=(4.5,))
else:
    OUT=ROOT/'notebooks'/'Signed_Jumps'/'01'
OUT.mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
if not SYNTHETIC:
    chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_signed_jumps.py')],cwd=str(ROOT),env=env,capture_output=True,text=True)
    print(chk.stdout[-600:]);print(chk.stderr[-400:]);assert chk.returncode==0,'tests failed'
print(json.dumps(sj.registration(p),indent=1)[:1800])

## Data and volatility engine

In [ ]:
T0=time.time()
if SYNTHETIC:
    S={a:s for a,s in zip(p.assets,sj.simulate(n_days=40,n_assets=len(p.assets),eta=((0.2,0.0),(0.05,0.3)),seed=5,df=8).values())}
else:
    BIN=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent/'Binance'
    S={}
    for A in p.assets:
        f=BIN/'basis'/'spot'/f'{A}USDT_spot_1m.parquet'
        if not f.exists():print('MISSING',f);continue
        S[A]=sj.load_spot_1m(f)
assert len(S)==len(p.assets),f'assets loaded: {list(S)}'
assets,pan,dev_end=sj.prepare(S,p)
print('DEV/TEST boundary:',dev_end,'| panel rows',len(pan.r),f'| {time.time()-T0:.0f}s')
q=[]
for i,a in enumerate(assets):
    m=pan.asset_idx==i
    r=a.r;ok=np.isfinite(r)
    row={'asset':a.name,'first':str(a.time[0]),'last':str(a.time[-1]),'grid_bars':len(r),'observed_returns':int(ok.sum()),
         'zero_return_share':float((r[ok]==0).mean()),'panel_rows':int(m.sum()),
         'HAR_coef':np.round(a.cols['har_coef'],3).tolist()}
    for s in sj.SIGNS:
        for per,mm in (('dev',pan.dev),('test',~pan.dev)):
            row[f'{s}_{per}']=int(pan.J[s][m&mm].sum())
        row[f'{s}_per_day']=float(pan.J[s][m].sum()/(m.sum()/1440))
    q.append(row)
quality=pd.DataFrame(q);display(quality);quality.to_csv(OUT/'01_data_quality.csv',index=False)
sz=[]
for i,a in enumerate(assets):
    m=pan.asset_idx==i
    for s in sj.SIGNS:
        rr=np.abs(pan.r[m&(pan.J[s]>0)])*1e4
        zz=np.abs(pan.Z[m&(pan.J[s]>0)])
        sz.append({'asset':a.name,'sign':s,'n':len(rr),'mean_bp':rr.mean(),'median_bp':np.median(rr),'p90_bp':np.quantile(rr,.9),
                   'mean_absZ':zz.mean()})
sizes=pd.DataFrame(sz);display(sizes.round(2));sizes.to_csv(OUT/'01_jump_sizes.csv',index=False)

## Fit the ladder on DEV

In [ ]:
T1=time.time()
fits,Ecache=sj.fit_ladder(assets,pan,p)
print(f'fit {time.time()-T1:.0f}s')
fr=[]
for (m,s),f in fits.items():
    names=sj.design_names(pan,m,p)
    row={'model':m,'sign':s,'half_life_min':f.half_life,'ll_dev':f.ll_dev,'n_dev':f.n_dev,'converged':f.converged}
    row.update({n:v for n,v in zip(names,f.b)})
    for k,v in enumerate(f.eta):row[f'eta{k}']=v
    fr.append(row)
fit_tab=pd.DataFrame(fr);display(fit_tab.round(4));fit_tab.to_csv(OUT/'01_fits.csv',index=False)
assert fit_tab.converged.all() or SYNTHETIC,'non-converged fit'
eta=sj.eta_matrix(fits)
# cluster(day)-robust SEs on DEV for (d) and (e)
dev=pan.dev;se_rows=[]
for m in ('d','e'):
    X=sj.design(pan,m)[dev]
    for s in sj.SIGNS:
        f=fits[(m,s)];Ex=sj.excitation_design(Ecache[f.half_life],m,s)[dev]
        se=sj.cluster_se(f,X.astype(float),Ex,pan.J[s][dev],pan.day[dev])
        names=sj.design_names(pan,m,p)+['eta_from_up','eta_from_down']
        for n,v,e in zip(names,np.concatenate([f.b,f.eta]),se):
            se_rows.append({'model':m,'child_sign':s,'param':n,'estimate':v,'cluster_se':e})
se_tab=pd.DataFrame(se_rows);se_tab.to_csv(OUT/'01_param_se.csv',index=False)
display(se_tab[se_tab.param.str.startswith(('eta','TSK','L_','F_','TV'))].round(4))
cont=eta.from_up[0]+eta.from_down[1]-eta.from_down[0]-eta.from_up[1]
print('eta matrix (d):');display(eta.round(4))
print(f"branching (c) {sj.branching(fits,'c'):.3f} | (d) {sj.branching(fits,'d'):.3f} | continuation index {cont:+.3f}")

## Gates G1, G2 on TEST

In [ ]:
cmp,njump=sj.compare(fits,pan,Ecache,p)
print('TEST jumps:',njump);display(cmp[['test','delta_per_row','q0125','q025','q975','n_days','status','bonferroni']])
cmp.to_csv(OUT/'01_tests.csv',index=False)

## G3: 15-minute signed jump forecast (5 and 60 secondary)

In [ ]:
sel=sj.selected_model(cmp);print('selected model:',sel)
T2=time.time()
g3,cal=sj.g3_table(fits,sel,assets,pan,p)
print(f'{time.time()-T2:.0f}s');display(g3.round(5));g3.to_csv(OUT/'01_g3.csv',index=False)
if len(cal):display(cal.round(4));cal.to_csv(OUT/'01_sign_calibration.csv',index=False)

## Sensitivity: jump threshold c (reported, never used for decisions)

In [ ]:
sens=[]
from dataclasses import replace
for cc in p.c_sensitivity:
    pc=replace(p,c=cc)
    sj.rejump(assets,pc);pn=sj.build_panel(assets,pc)
    fc,Ec=sj.fit_ladder(assets,pn,pc,models=('a','b','c','d'))
    cm,nj=sj.compare(fc,pn,Ec,pc)
    for _,r in cm.iterrows():
        sens.append({'c':cc,'test':r.test,'delta_per_row':r.delta_per_row,'q025':r.q025,'status':r.status,
                     'jumps_up':nj['up'],'jumps_down':nj['down'],'rho_c':sj.branching(fc,'c'),'rho_d':sj.branching(fc,'d')})
sj.rejump(assets,p)
sens=pd.DataFrame(sens);display(sens.round(6));sens.to_csv(OUT/'01_c_sensitivity.csv',index=False)

## Decision

In [ ]:
dec=sj.decide(cmp,g3,p)
dec.update(branching_c=sj.branching(fits,'c'),branching_d=sj.branching(fits,'d'),continuation_index=float(cont),
           eta_matrix=eta.to_dict(orient='records'),test_jumps=njump,dev_end=str(dev_end),synthetic=SYNTHETIC)
print(json.dumps(dec,indent=1,default=float))
if not SYNTHETIC:
    (OUT/'01_decision.json').write_text(json.dumps(dec,indent=1,default=float))
    (OUT/'01_manifest.json').write_text(json.dumps({'registration':sj.registration(p),'runtime_s':time.time()-T0,
        'assets':list(S),'rows':int(len(pan.r))},indent=1,default=str))
print(f'total {time.time()-T0:.0f}s')